# 01 - CultPass external database

Creates `data/external/cultpass.db`, the customer's own system that the UDA-Hub
tools abstract (users, subscriptions, experiences, reservations).

Changes from the starter, all marked in the cells below:
* `random.seed(42)` and explicit subscription plans, so the demo scenarios in
  `03_agentic_app` behave the same on every run;
* reservations for more users (the starter's TODO), including one inside the
  24-hour cancellation window and one cancelled reservation;
* one sold-out experience, to exercise the waitlist path.

In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Reproducible runs: the starter drew slots/plans at random on every run.
random.seed(42)

# Udahub Accounts

## Cultpass Database

**Init DB**

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
2026-09-27 02:20:21,245 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-09-27 02:20:21,250 INFO sqlalchemy.engine.Engine COMMIT


✅ Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [6]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [7]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [8]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [9]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [10]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [11]:

# Explicit plans instead of random.choice so every scenario is reproducible.
# (status, tier, monthly_quota)
plan_profiles = {
    "a4ab87": ("active", "premium", 8),     # Alice Kingsley - account is blocked
    "f556c0": ("active", "basic", 4),       # Bob Stone
    "88382b": ("active", "premium", 10),    # Cathy Bloom
    "888fb2": ("cancelled", "basic", 4),    # David Noir - account is blocked
    "f1f10d": ("paused", "basic", 4),       # Eva Green
    "e6376d": ("active", "premium", 6),     # Frank Ocean
}
cycle_start = datetime.now() - timedelta(days=12)

with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        status, tier, quota = plan_profiles[user_info["id"]]
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=status,
            tier=tier,
            monthly_quota=quota,
            started_at=cycle_start,
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)


**Reservation**

In [12]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [13]:

# More reservations (the starter's TODO), so several customers have history.
# Experiences are dated now+1 .. now+7 days in file order, which gives:
#   Bob   - Carnival tour (tomorrow: inside the 24h window) + Samba Night
#   Cathy - MASP visit (reserved) + Paddleboarding (already cancelled)
#   Frank - Christ the Redeemer
# and the Ibirapuera bike ride is sold out (waitlist scenario).
def exp_id(session, title):
    return session.query(cultpass.Experience).filter_by(title=title).one().experience_id

more_reservations = [
    ("f556c0", "Carnival History Tour in Olinda", "reserved"),
    ("f556c0", "Samba Night at Lapa", "reserved"),
    ("88382b", "Modern Art at MASP", "reserved"),
    ("88382b", "Sunset Paddleboarding", "cancelled"),
    ("e6376d", "Christ the Redeemer Experience", "reserved"),
]

with get_session(engine) as session:
    for user_id, title, status in more_reservations:
        session.add(cultpass.Reservation(
            reservation_id=str(uuid.uuid4())[:6],
            user_id=user_id,
            experience_id=exp_id(session, title),
            status=status,
            created_at=datetime.now() - timedelta(days=2),
        ))
    sold_out = session.query(cultpass.Experience).filter_by(title="Ibirapuera Park Bike Ride").one()
    sold_out.slots_available = 0


# Tests

In [14]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [15]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='f7e2b5', user_id='a4ab87', status='active', tier='premium')>
<Subscription(subscription_id='d64846', user_id='f556c0', status='active', tier='basic')>
<Subscription(subscription_id='189059', user_id='88382b', status='active', tier='premium')>
<Subscription(subscription_id='c5a829', user_id='888fb2', status='cancelled', tier='basic')>
<Subscription(subscription_id='58035b', user_id='f1f10d', status='paused', tier='basic')>
<Subscription(subscription_id='026149', user_id='e6376d', status='active', tier='premium')>


In [16]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='2125f1', title='Carnival History Tour in Olinda', when='2026-09-28 02:20:22.981084')>
<Experience(experience_id='3cf726', title='Sunset Paddleboarding', when='2026-09-29 02:20:23.000337')>
<Experience(experience_id='c19c45', title='Pelourinho Colonial Walk', when='2026-09-30 02:20:23.001335')>
<Experience(experience_id='60905c', title='Samba Night at Lapa', when='2026-10-01 02:20:23.001335')>
<Experience(experience_id='1bba22', title='Christ the Redeemer Experience', when='2026-10-02 02:20:23.001335')>
<Experience(experience_id='b21900', title='Modern Art at MASP', when='2026-10-03 02:20:23.001335')>
<Experience(experience_id='f099df', title='Ibirapuera Park Bike Ride', when='2026-10-04 02:20:23.001335')>


In [17]:

with get_session(engine) as session:
    for r in session.query(cultpass.Reservation).all():
        print(r.user.full_name, "|", r.experience.title, "|", r.experience.when.strftime("%Y-%m-%d %H:%M"), "|", r.status)
    print("tables:", sorted(cultpass.Base.metadata.tables))


Alice Kingsley | Sunset Paddleboarding | 2026-09-29 02:20 | reserved
Alice Kingsley | Modern Art at MASP | 2026-10-03 02:20 | reserved
Bob Stone | Carnival History Tour in Olinda | 2026-09-28 02:20 | reserved
Bob Stone | Samba Night at Lapa | 2026-10-01 02:20 | reserved
Cathy Bloom | Modern Art at MASP | 2026-10-03 02:20 | reserved
Cathy Bloom | Sunset Paddleboarding | 2026-09-29 02:20 | cancelled
Frank Ocean | Christ the Redeemer Experience | 2026-10-02 02:20 | reserved
tables: ['experiences', 'reservations', 'subscriptions', 'users']
